# BTC ↔ Polymarket Correlation Analysis

This notebook analyzes the lead/lag relationship between BTC price movements
and Polymarket crypto prediction market ("Bitcoin above $Xk") probability changes.

**Goals:**
1. Load BTC 10-min OHLCV bars and Polymarket YES token price bars
2. Compute cross-correlation (CCF) at various lags
3. Analyze delta sensitivity by strike (moneyness)
4. Visualize results for strategy parameter tuning

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Data root relative to notebook location
DATA_ROOT = os.path.join(os.path.dirname(os.getcwd()), 'Data')
if not os.path.exists(DATA_ROOT):
    DATA_ROOT = os.path.join(os.getcwd(), '..', 'Data')
print(f'Data root: {DATA_ROOT}')

## 1. Load BTC Reference Data

In [ ]:
def load_ohlcv_csvs(directory, date_prefix=None):
    """Load all *_trade.csv files from a directory into a single DataFrame."""
    pattern = os.path.join(directory, '*_trade.csv')
    files = sorted(glob.glob(pattern))
    if not files:
        return pd.DataFrame()
    
    dfs = []
    for f in files:
        basename = os.path.basename(f)
        date_str = basename.split('_')[0]  # e.g., '20260224'
        date = pd.Timestamp(date_str)
        
        df = pd.read_csv(f, header=None,
                         names=['ms_from_midnight', 'open', 'high', 'low', 'close', 'volume'])
        df['timestamp'] = date + pd.to_timedelta(df['ms_from_midnight'], unit='ms')
        dfs.append(df)
    
    result = pd.concat(dfs, ignore_index=True)
    result = result.sort_values('timestamp').set_index('timestamp')
    return result

# Load BTC data
btc_dir = os.path.join(DATA_ROOT, 'reference', 'btc-usd')
if os.path.exists(btc_dir):
    btc = load_ohlcv_csvs(btc_dir)
    print(f'BTC bars: {len(btc)}, range: {btc.index.min()} to {btc.index.max()}')
    btc[['close']].plot(title='BTC/USDT Close Price')
    plt.ylabel('Price (USDT)')
    plt.show()
else:
    print(f'BTC data not found at {btc_dir}')
    print('Run: dotnet run -- --download-btc --days 7')
    btc = pd.DataFrame()

## 2. Load Polymarket YES Token Data

In [ ]:
def extract_strike(dirname):
    """Extract strike price from directory name like 'bitcoin-above-74k-on-march-3yes'."""
    match = re.search(r'(\d+)k', dirname)
    if match:
        return int(match.group(1)) * 1000
    return None

# Find all bitcoin-above-*yes token directories
poly_dir = os.path.join(DATA_ROOT, 'crypto', 'polymarket', 'minute')
token_dirs = sorted(glob.glob(os.path.join(poly_dir, 'bitcoin-above-*yes')))

tokens = {}
for d in token_dirs:
    name = os.path.basename(d)
    strike = extract_strike(name)
    if strike is None:
        continue
    df = load_ohlcv_csvs(d)
    if len(df) > 0:
        tokens[strike] = df
        print(f'  {name}: strike=${strike:,}, bars={len(df)}')

print(f'\nLoaded {len(tokens)} YES tokens with strikes: {sorted(tokens.keys())}')

## 3. Resample & Align (10-minute forward-fill)

In [ ]:
if len(btc) > 0 and len(tokens) > 0:
    # Resample BTC to 10-min close
    btc_10m = btc['close'].resample('10min').last().ffill()
    btc_ret = btc_10m.pct_change().dropna()
    
    # Resample each token to 10-min close
    token_10m = {}
    token_ret = {}
    for strike, df in tokens.items():
        close = df['close'].resample('10min').last().ffill()
        token_10m[strike] = close
        token_ret[strike] = close.pct_change().dropna()
    
    print(f'BTC 10-min returns: {len(btc_ret)}')
    for strike in sorted(token_ret.keys()):
        print(f'  ${strike:,} YES returns: {len(token_ret[strike])}')
else:
    print('Insufficient data for alignment. Ensure both BTC and Polymarket data are downloaded.')

## 4. Cross-Correlation Function (CCF)

In [ ]:
def compute_ccf(x, y, max_lag=6):
    """Compute cross-correlation at lags from -max_lag to +max_lag.
    Positive lag means x leads y."""
    # Align on common index
    common = x.index.intersection(y.index)
    if len(common) < 10:
        return [], []
    x_aligned = x.loc[common].values
    y_aligned = y.loc[common].values
    
    lags = list(range(-max_lag, max_lag + 1))
    correlations = []
    
    for lag in lags:
        if lag > 0:
            r, _ = stats.pearsonr(x_aligned[:-lag], y_aligned[lag:])
        elif lag < 0:
            r, _ = stats.pearsonr(x_aligned[-lag:], y_aligned[:lag])
        else:
            r, _ = stats.pearsonr(x_aligned, y_aligned)
        correlations.append(r)
    
    return lags, correlations

if len(btc_ret) > 0 and len(token_ret) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    strikes = sorted(token_ret.keys())
    for i, strike in enumerate(strikes[:6]):
        ax = axes[i]
        lags, corrs = compute_ccf(btc_ret, token_ret[strike], max_lag=6)
        
        colors = ['green' if c > 0 else 'red' for c in corrs]
        ax.bar(lags, corrs, color=colors, alpha=0.7)
        ax.axhline(y=0, color='black', linewidth=0.5)
        ax.set_title(f'BTC vs ${strike//1000}k YES (CCF)')
        ax.set_xlabel('Lag (10-min periods, + = BTC leads)')
        ax.set_ylabel('Correlation')
        ax.set_ylim(-1, 1)
    
    plt.tight_layout()
    plt.suptitle('Cross-Correlation: BTC Returns vs Polymarket YES Token Returns', y=1.02)
    plt.show()
else:
    print('Skipping CCF — insufficient data')

## 5. Delta Sensitivity by Strike (Moneyness)

In [ ]:
if len(btc_ret) > 0 and len(token_ret) > 0:
    # Use the latest BTC price to compute moneyness
    btc_latest = btc_10m.iloc[-1] if len(btc_10m) > 0 else 85000
    print(f'Latest BTC price: ${btc_latest:,.0f}')
    
    results = []
    for strike in sorted(token_ret.keys()):
        moneyness = (btc_latest - strike) / btc_latest
        
        # Correlation at lag=0
        common = btc_ret.index.intersection(token_ret[strike].index)
        if len(common) < 10:
            continue
        r, p = stats.pearsonr(btc_ret.loc[common], token_ret[strike].loc[common])
        
        # Best lag
        lags, corrs = compute_ccf(btc_ret, token_ret[strike], max_lag=6)
        best_lag = lags[np.argmax(np.abs(corrs))]
        best_corr = corrs[np.argmax(np.abs(corrs))]
        
        results.append({
            'strike': strike,
            'moneyness': moneyness,
            'corr_lag0': r,
            'p_value': p,
            'best_lag': best_lag,
            'best_corr': best_corr
        })
    
    df_results = pd.DataFrame(results)
    print('\n=== Delta Sensitivity by Strike ===')
    print(df_results.to_string(index=False, float_format='{:.4f}'.format))
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.scatter(df_results['moneyness'], df_results['corr_lag0'], 
                s=100, c='steelblue', edgecolors='navy')
    for _, row in df_results.iterrows():
        ax1.annotate(f"${row['strike']//1000:.0f}k", 
                     (row['moneyness'], row['corr_lag0']),
                     textcoords='offset points', xytext=(5, 5), fontsize=8)
    ax1.axhline(y=0, color='gray', linestyle='--')
    ax1.axvline(x=0, color='gray', linestyle='--', label='ATM')
    ax1.set_xlabel('Moneyness (BTC - Strike) / BTC')
    ax1.set_ylabel('Pearson Correlation (lag=0)')
    ax1.set_title('Correlation vs Moneyness')
    ax1.legend()
    
    ax2.bar(df_results['strike'] / 1000, df_results['best_corr'], 
            color='steelblue', alpha=0.7)
    ax2.set_xlabel('Strike ($k)')
    ax2.set_ylabel('Best Correlation (any lag)')
    ax2.set_title('Peak Correlation by Strike')
    
    plt.tight_layout()
    plt.show()
else:
    print('Skipping delta analysis — insufficient data')

## 6. Rolling Correlation

In [ ]:
if len(btc_ret) > 0 and len(token_ret) > 0:
    # Pick the token closest to ATM
    btc_latest = btc_10m.iloc[-1] if len(btc_10m) > 0 else 85000
    closest_strike = min(token_ret.keys(), key=lambda s: abs(s - btc_latest))
    
    common = btc_ret.index.intersection(token_ret[closest_strike].index)
    if len(common) >= 20:
        aligned = pd.DataFrame({
            'btc': btc_ret.loc[common],
            'token': token_ret[closest_strike].loc[common]
        })
        
        rolling_corr = aligned['btc'].rolling(20).corr(aligned['token'])
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
        
        ax1.plot(aligned.index, aligned['btc'], alpha=0.7, label='BTC return')
        ax1.plot(aligned.index, aligned['token'], alpha=0.7, label=f'${closest_strike//1000}k YES return')
        ax1.legend()
        ax1.set_ylabel('10-min Return')
        ax1.set_title('BTC vs ATM Token Returns')
        
        ax2.plot(rolling_corr.index, rolling_corr, color='purple', alpha=0.8)
        ax2.axhline(y=0.3, color='green', linestyle='--', label='Signal threshold (+0.3)')
        ax2.axhline(y=-0.3, color='red', linestyle='--', label='Signal threshold (-0.3)')
        ax2.axhline(y=0, color='gray', linewidth=0.5)
        ax2.fill_between(rolling_corr.index, 0.3, rolling_corr.clip(lower=0.3), alpha=0.2, color='green')
        ax2.fill_between(rolling_corr.index, -0.3, rolling_corr.clip(upper=-0.3), alpha=0.2, color='red')
        ax2.set_ylabel('Rolling Correlation (20-period)')
        ax2.set_title(f'Rolling Correlation: BTC vs ${closest_strike//1000}k YES')
        ax2.legend()
        ax2.set_ylim(-1, 1)
        
        plt.tight_layout()
        plt.show()
        
        # Summary statistics
        valid = rolling_corr.dropna()
        print(f'\nRolling correlation stats (ATM strike ${closest_strike//1000}k):')
        print(f'  Mean:   {valid.mean():.4f}')
        print(f'  Std:    {valid.std():.4f}')
        print(f'  % > 0.3: {(valid > 0.3).mean()*100:.1f}%')
        print(f'  % < -0.3: {(valid < -0.3).mean()*100:.1f}%')
    else:
        print(f'Insufficient overlapping data for rolling correlation (need 20, have {len(common)})')
else:
    print('Skipping rolling correlation — insufficient data')

## 7. Scatter: BTC Return vs Token Return (lag=0)

In [ ]:
if len(btc_ret) > 0 and len(token_ret) > 0:
    btc_latest = btc_10m.iloc[-1] if len(btc_10m) > 0 else 85000
    
    # Pick 4 representative strikes: deep ITM, ATM, slight OTM, deep OTM
    strikes_sorted = sorted(token_ret.keys())
    picks = []
    if len(strikes_sorted) >= 4:
        picks = [strikes_sorted[0], 
                 min(strikes_sorted, key=lambda s: abs(s - btc_latest)),
                 strikes_sorted[-2] if len(strikes_sorted) > 2 else strikes_sorted[-1],
                 strikes_sorted[-1]]
        picks = list(dict.fromkeys(picks))  # deduplicate preserving order
    else:
        picks = strikes_sorted
    
    fig, axes = plt.subplots(1, len(picks), figsize=(5*len(picks), 4))
    if len(picks) == 1:
        axes = [axes]
    
    for i, strike in enumerate(picks):
        ax = axes[i]
        common = btc_ret.index.intersection(token_ret[strike].index)
        if len(common) < 5:
            continue
        x = btc_ret.loc[common]
        y = token_ret[strike].loc[common]
        
        ax.scatter(x, y, alpha=0.3, s=10)
        # Regression line
        slope, intercept, r, p, se = stats.linregress(x, y)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2)
        
        moneyness = (btc_latest - strike) / btc_latest
        ax.set_title(f"${strike//1000}k (m={moneyness:.2f})\nr={r:.3f}, slope={slope:.2f}")
        ax.set_xlabel('BTC 10m Return')
        ax.set_ylabel('Token 10m Return')
    
    plt.suptitle('Scatter: BTC Return vs YES Token Return (lag=0)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Skipping scatter — insufficient data')

## Summary

Key findings to inform `BtcFollowMMStrategy` parameters:

1. **Lead/lag**: Check the CCF plots — if BTC leads by 1-2 periods (10-20min), the strategy has predictive value
2. **Delta sensitivity**: ATM tokens (moneyness ~ 0) should show strongest correlation; deep ITM/OTM should be weak
3. **Correlation stability**: If rolling correlation is often > 0.3, the signal is reliable; if unstable, reduce `MomentumSpreadMultiplier`
4. **Suggested parameters**:
   - `MomentumThreshold`: ~0.002 (0.2%) — adjust based on BTC return distribution
   - `MinCorrelation`: 0.3 — confirmed by rolling correlation analysis
   - `MomentumSpreadMultiplier`: 2.0 — scale up if correlation is consistently strong
   - `MomentumSizeReduction`: 0.5 — more aggressive (0.3) if delta is high